***INFERENCE AND SUBMISSION***

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import timm
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d

device = torch.device('cpu')
base_path = '/kaggle/input/competitions/birdclef-2026'

path_model_1 = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_final_phaseB_EN.pth'
path_model_2 = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_final_phaseB_resnet34d.pth' 

train_csv = os.path.join(base_path, 'train.csv')
df = pd.read_csv(train_csv)
label_encoder = LabelEncoder()
label_encoder.fit(df['primary_label'])
num_classes = len(label_encoder.classes_)

class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=128, n_fft=2048, hop_length=512, f_min=40, f_max=15000):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length, f_min=f_min, f_max=f_max, n_mels=n_mels, power=2.0 
        )
        self.eps, self.s, self.alpha, self.delta, self.r = 1e-6, 0.025, 0.98, 2.0, 0.5
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=24)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=64)

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        ema = x.clone()
        for t in range(1, x.size(-1)):
            ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
        x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        x = (x - x.mean()) / (x.std() + 1e-6)
        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        return x

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        return torch.sum(att_weights * frame_logits, dim=-1), frame_logits

class BirdSED_EfficientNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.audio_extractor = AudioToSpectrogramGPU(n_mels=128)
        self.backbone = timm.create_model('efficientnet_b0', pretrained=False, in_chans=1)
        self.backbone.classifier = nn.Identity()
        self.backbone.global_pool = nn.Identity()
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.sed_head = AttentivePooling(in_channels=1280, num_classes=num_classes)

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        x = self.backbone.forward_features(x)
        x = self.freq_pool(x).squeeze(2)
        clip_logits, _ = self.sed_head(x)
        return clip_logits

class BirdSED_ResNet34d(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.audio_extractor = AudioToSpectrogramGPU(n_mels=128)
        self.backbone = timm.create_model('resnet34d', pretrained=False, in_chans=1)
        self.backbone.fc = nn.Identity()
        self.backbone.global_pool = nn.Identity()
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.sed_head = AttentivePooling(in_channels=512, num_classes=num_classes)

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        x = self.backbone.forward_features(x)
        x = self.freq_pool(x).squeeze(2)
        clip_logits, _ = self.sed_head(x)
        return clip_logits

model1 = BirdSED_EfficientNet(num_classes=num_classes)
model1.load_state_dict(torch.load(path_model_1, map_location=device))
model1.to(device).eval()

model2 = BirdSED_ResNet34d(num_classes=num_classes)
model2.load_state_dict(torch.load(path_model_2, map_location=device))
model2.to(device).eval()

def get_padded_chunk(wf, start, end, chunk_samples):
    c_len = end - start
    pad_left = max(0, -start)
    start_safe = max(0, start)
    end_safe = min(end, wf.shape[1])
    pad_right = max(0, end - wf.shape[1])
    chunk = wf[:, start_safe:end_safe]
    if pad_left > 0 or pad_right > 0:
        chunk = torch.nn.functional.pad(chunk, (pad_left, pad_right))
    return chunk

def apply_temporal_smoothing(probs_matrix, sigma=1.0):
    smoothed_probs = np.zeros_like(probs_matrix)
    for class_idx in range(probs_matrix.shape[1]):
        smoothed_probs[:, class_idx] = gaussian_filter1d(probs_matrix[:, class_idx], sigma=sigma)
    return smoothed_probs

sample_df = pd.read_csv(os.path.join(base_path, 'sample_submission.csv'))
expected_species_columns = sample_df.columns[1:] 
test_audio_dir = os.path.join(base_path, 'test_soundscapes')
predictions_dict = {}

predicted_species_names = label_encoder.inverse_transform(range(num_classes))
species_to_idx = {sp: idx for idx, sp in enumerate(predicted_species_names)}

if os.path.exists(test_audio_dir):
    test_files = [f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]
    print(f"Processing {len(test_files)} files dynamically with TTA Ensemble...")
    
    for file in test_files:
        file_path = os.path.join(test_audio_dir, file)
        file_stem = file.replace('.ogg', '')
        
        try:
            waveform, sr = torchaudio.load(file_path)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            chunk_samples = 5 * 32000
            shift_samples = int(0.5 * 32000)
            num_chunks = waveform.shape[1] // chunk_samples 
            
            file_probs = []
            batch_size = 16 
            
            for b in range(0, num_chunks, batch_size):
                wave_chunks_base = []
                wave_chunks_left = []
                wave_chunks_right = []
                
                for i in range(b, min(b + batch_size, num_chunks)):
                    start = i * chunk_samples
                    end = start + chunk_samples
                    wave_chunks_base.append(get_padded_chunk(waveform, start, end, chunk_samples))
                    wave_chunks_left.append(get_padded_chunk(waveform, start - shift_samples, end - shift_samples, chunk_samples))
                    wave_chunks_right.append(get_padded_chunk(waveform, start + shift_samples, end + shift_samples, chunk_samples))
                    
                batch_base = torch.stack(wave_chunks_base).to(device)
                batch_left = torch.stack(wave_chunks_left).to(device)
                batch_right = torch.stack(wave_chunks_right).to(device)
                
                with torch.no_grad():
                    p1_base = torch.sigmoid(model1(batch_base)).cpu().numpy()
                    p1_left = torch.sigmoid(model1(batch_left)).cpu().numpy()
                    p1_right = torch.sigmoid(model1(batch_right)).cpu().numpy()
                    p1_ensemble = (p1_base + p1_left + p1_right) / 3.0
                    
                    p2_base = torch.sigmoid(model2(batch_base)).cpu().numpy()
                    p2_left = torch.sigmoid(model2(batch_left)).cpu().numpy()
                    p2_right = torch.sigmoid(model2(batch_right)).cpu().numpy()
                    p2_ensemble = (p2_base + p2_left + p2_right) / 3.0

                    final_ensemble = (p1_ensemble + p2_ensemble) / 2.0
                    file_probs.append(final_ensemble)
            
            if len(file_probs) > 0:
                file_probs = np.vstack(file_probs)
                smoothed_probs = apply_temporal_smoothing(file_probs, sigma=1.0)
                for i in range(num_chunks):
                    row_id = f"{file_stem}_{(i + 1) * 5}"
                    predictions_dict[row_id] = smoothed_probs[i] 
                    
            del waveform
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {file}: {e}")

for idx, row in sample_df.iterrows():
    r_id = row['row_id']
    if r_id in predictions_dict:
        pred_probs = predictions_dict[r_id]
        for col in expected_species_columns:
            if col in species_to_idx:
                sample_df.at[idx, col] = pred_probs[species_to_idx[col]]

sample_df.to_csv('submission.csv', index=False)
print("Submission.csv generated")